Merging data from the same meeting

In [2]:
from pathlib import Path
import pandas as pd

# loading data

speed_dating_csv_path = next(Path("data").rglob("Speed Dating Data.csv"))
speed_dating_data = pd.read_csv(speed_dating_csv_path, encoding="latin-1")

speed_dating_data.head()

,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


In [3]:
# merging data from the same meeting

woman_rows = speed_dating_data[speed_dating_data["gender"] == 0]
man_rows = speed_dating_data[speed_dating_data["gender"] == 1]

meetings = woman_rows.merge(
    man_rows,
    left_on=["iid", "pid"],
    right_on=["pid", "iid"],
    suffixes=("_woman", "_man"),
)

meetings.head()

,iid_woman,id_woman,gender_woman,idg_woman,condtn_woman,wave_woman,round_woman,position_woman,positin1_woman,order_woman,...,attr3_3_man,sinc3_3_man,intel3_3_man,fun3_3_man,amb3_3_man,attr5_3_man,sinc5_3_man,intel5_3_man,fun5_3_man,amb5_3_man
0,1,1.0,0,1,1,1,10,7,NaN,4,...,7.0,8.0,7.0,7.0,5.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,9.0,9.0,9.0,9.0,9.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,7.0,7.0,9.0,7.0,9.0,NaN,NaN,NaN,NaN,NaN


Spliting features and targets and removing after decision features

In [4]:
meetings = meetings.copy()

woman_wants_another_meeting = meetings["dec_woman"] == 1
man_wants_another_meeting = meetings["dec_man"] == 1

meetings["outcome"] = 4
meetings.loc[woman_wants_another_meeting & man_wants_another_meeting, "outcome"] = 1
meetings.loc[woman_wants_another_meeting & ~man_wants_another_meeting, "outcome"] = 2
meetings.loc[~woman_wants_another_meeting & man_wants_another_meeting, "outcome"] = 3

def column_without_person_suffix(column):
    if column.endswith("_woman"):
        return column.removesuffix("_woman")
    if column.endswith("_man"):
        return column.removesuffix("_man")
    return column

def recorded_after_the_decision(column):
    original_name = column_without_person_suffix(column)
    return original_name.endswith(("_s", "_2", "_3")) or original_name in {"length", "you_call", "them_cal"}

scorecard_columns_written_together_with_the_decision = {
    "dec", "dec_o", "match", "match_es",
    "attr", "sinc", "intel", "fun", "amb", "shar", "like", "prob", "met",
    "attr_o", "sinc_o", "intel_o", "fun_o", "amb_o", "shar_o", "like_o", "prob_o", "met_o",
}

columns_to_drop = [
    column
    for column in meetings.columns
    if recorded_after_the_decision(column)
    or column_without_person_suffix(column) in scorecard_columns_written_together_with_the_decision
]

meetings = meetings.drop(columns=columns_to_drop)

print("dropped", len(columns_to_drop))
print("shape", meetings.shape)
print(meetings["outcome"].value_counts(dropna=False).sort_index())

meetings.head()

dropped 218
shape (4184, 173)
outcome
1     690
2     839
3    1296
4    1359
Name: count, dtype: int64


,iid_woman,id_woman,gender_woman,idg_woman,condtn_woman,wave_woman,round_woman,position_woman,positin1_woman,order_woman,...,sinc3_1_man,fun3_1_man,intel3_1_man,amb3_1_man,attr5_1_man,sinc5_1_man,intel5_1_man,fun5_1_man,amb5_1_man,outcome
0,1,1.0,0,1,1,1,10,7,NaN,4,...,9.0,7.0,8.0,5.0,NaN,NaN,NaN,NaN,NaN,2
1,1,1.0,0,1,1,1,10,7,NaN,3,...,9.0,9.0,10.0,9.0,NaN,NaN,NaN,NaN,NaN,2
2,1,1.0,0,1,1,1,10,7,NaN,10,...,7.0,8.0,8.0,3.0,NaN,NaN,NaN,NaN,NaN,1
3,1,1.0,0,1,1,1,10,7,NaN,5,...,9.0,9.0,9.0,9.0,NaN,NaN,NaN,NaN,NaN,1
4,1,1.0,0,1,1,1,10,7,NaN,7,...,7.0,7.0,9.0,9.0,NaN,NaN,NaN,NaN,NaN,1


Checking lacks and ionvalid values in the data

In [5]:
empty_cells = int(meetings.isna().sum().sum())
all_cells = meetings.size
print("empty cells", empty_cells)
print("all cells", all_cells)
print("empty share", empty_cells / all_cells)

empty_share_by_column = meetings.isna().mean().sort_values(ascending=False)
display(empty_share_by_column[empty_share_by_column >= 0.2])

empty cells 60532
all cells 723832
empty share 0.08362713999933687


expnum_woman      0.791587
expnum_man        0.780593
mn_sat_man        0.694073
tuition_man       0.647467
income_man        0.572897
mn_sat_woman      0.557122
tuition_woman     0.496176
attr5_1_man       0.415153
fun5_1_man        0.415153
sinc5_1_man       0.415153
amb5_1_man        0.415153
intel5_1_man      0.415153
undergra_man      0.415153
amb5_1_woman      0.412285
intel5_1_woman    0.412285
fun5_1_woman      0.412285
attr5_1_woman     0.412285
sinc5_1_woman     0.412285
undergra_woman    0.410373
income_woman      0.405832
shar4_1_woman     0.230163
intel4_1_woman    0.224904
fun4_1_woman      0.224904
sinc4_1_woman     0.224904
attr4_1_woman     0.224904
amb4_1_woman      0.224904
amb4_1_man        0.224187
intel4_1_man      0.224187
fun4_1_man        0.224187
shar4_1_man       0.224187
sinc4_1_man       0.224187
attr4_1_man       0.224187
positin1_man      0.219407
positin1_woman    0.219407
dtype: float64

poprawiane tych samych wartości zapisanych inaczej lub z błędem dla fech tekstowych

In [6]:
import re

text_columns = [
    "field_woman", "field_man",
    "career_woman", "career_man",
    "from_woman", "from_man",
    "undergra_woman", "undergra_man",
]

def normalized_text(value):
    if pd.isna(value):
        return value
    value = str(value).strip().lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

for column in text_columns:
    meetings[column] = meetings[column].map(normalized_text)

In [7]:
from collections import defaultdict
from difflib import SequenceMatcher

def normalized_values(column):
    return {normalized_text(value) for value in meetings[column].dropna()}

groups = {
    "field": ["field_woman", "field_man"],
    "career": ["career_woman", "career_man"],
    "from": ["from_woman", "from_man"],
    "undergra": ["undergra_woman", "undergra_man"],
}

for group_name, columns in groups.items():
    keys = sorted(set().union(*(normalized_values(column) for column in columns)))
    print(group_name)
    for i, left in enumerate(keys):
        for right in keys[i + 1:]:
            if abs(len(left) - len(right)) > 12:
                continue
            if SequenceMatcher(None, left, right).ratio() >= 0.9:
                print(" ", left, " ~ ", right)

field
  biomedical informatics  ~  medical informatics
  finanace  ~  finance
  international affairs  ~  intrernational affairs
  master in public administration  ~  masters in public administration
  nutrition  ~  nutritiron
  speech language pathology  ~  speech languahe pathology
career
  acadeic  ~  academic
  academic  ~  academics
  clinical psychologist  ~  clinical psychology
  don t know  ~  i don t know
  enterpreneur  ~  entrepreneur
  entrepeneur  ~  entrepreneur
  international development  ~  international development work
  journalism  ~  journalist
  management consultant  ~  management consulting
  research academia  ~  researcher academia
  social work  ~  social worker
from
  cincinnati oh  ~  cincinnati ohio
  milan italy  ~  milano italy
  san francisco ca  ~  san francisco la
  washington d c  ~  washington dc
undergra
  american university  ~  the american university
  columbia  ~  columbiau
  columbia college  ~  columbia college cu
  connecticut college  ~  co

In [8]:
same_meaning = {
    "finanace": "finance",
    "nutritiron": "nutrition",
    "speech languahe pathology": "speech language pathology",
    "master in public administration": "masters in public administration",
    "intrernational affairs": "international affairs",
    "finance and economics": "finance economics",
    "ma biotechnology": "biotechnology",
    "mfa creative writing": "creative writing",
    "mathematics phd": "mathematics",
    "operations research seas": "operations research",
    "sipa international affairs": "international affairs",
    "master of international affairs": "international affairs",
    "mba master of international affairs sipa": "international affairs",
    "acadeic": "academic",
    "academics": "academic",
    "academia": "academic",
    "academician": "academic",
    "enterpreneur": "entrepreneur",
    "entrepeneur": "entrepreneur",
    "entrepreneurship": "entrepreneur",
    "research academia": "researcher academia",
    "researcher": "research",
    "clinical psychology": "clinical psychologist",
    "international development work": "international development",
    "journalist": "journalism",
    "management consultant": "management consulting",
    "general management consulting": "management consulting",
    "social work": "social worker",
    "corporate lawyer": "corporate law",
    "educator": "education",
    "engineering": "engineer",
    "investment banker": "investment banking",
    "don t know": "i don t know",
    "am not sure": "not sure",
    "conneticut college": "connecticut college",
    "u c berkeley": "uc berkeley",
    "columbiau": "columbia",
    "columbia college cu": "columbia college",
    "princeton u": "princeton",
    "univeristy of michigan": "university of michigan",
    "the american university": "american university",
    "washington d c": "washington dc",
    "milano italy": "milan italy",
    "cincinnati oh": "cincinnati ohio",
}

text_columns = [
    "field_woman", "field_man",
    "career_woman", "career_man",
    "from_woman", "from_man",
    "undergra_woman", "undergra_man",
]

for column in text_columns:
    meetings[column] = meetings[column].map(normalized_text).replace(same_meaning)

In [9]:
from collections import defaultdict
from difflib import SequenceMatcher

def normalized_values(column):
    return {normalized_text(value) for value in meetings[column].dropna()}

groups = {
    "field": ["field_woman", "field_man"],
    "career": ["career_woman", "career_man"],
    "from": ["from_woman", "from_man"],
    "undergra": ["undergra_woman", "undergra_man"],
}

for group_name, columns in groups.items():
    keys = sorted(set().union(*(normalized_values(column) for column in columns)))
    print(group_name)
    for i, left in enumerate(keys):
        for right in keys[i + 1:]:
            if abs(len(left) - len(right)) > 12:
                continue
            if SequenceMatcher(None, left, right).ratio() >= 0.9:
                print(" ", left, " ~ ", right)

field
  biomedical informatics  ~  medical informatics
career
from
  san francisco ca  ~  san francisco la
undergra
  northwestern university  ~  southwestern university


poprawianie wartości liczbowych 

In [10]:
numeric_text_columns = [
    "mn_sat_woman", "mn_sat_man",
    "tuition_woman", "tuition_man",
    "income_woman", "income_man",
]

for column in numeric_text_columns:
    meetings[column] = pd.to_numeric(
        meetings[column].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )

In [11]:
print(meetings[numeric_text_columns].dtypes)
display(meetings[numeric_text_columns].describe())

mn_sat_woman     float64
mn_sat_man       float64
tuition_woman    float64
tuition_man      float64
income_woman     float64
income_man       float64
dtype: object


,mn_sat_woman,mn_sat_man,tuition_woman,tuition_man,income_woman,income_man
count,1853.000000,1280.000000,2108.000000,1475.000000,2486.000000,1787.00000
mean,1284.722612,1321.272656,20885.072581,21589.170847,44372.644811,45601.84723
std,119.932030,116.299329,6633.580033,6890.926239,17443.086862,16852.84965
min,914.000000,990.000000,2406.000000,3730.000000,8607.000000,12063.00000
25%,1210.000000,1260.000000,15162.000000,15083.000000,31516.000000,31560.00000
50%,1290.000000,1340.000000,24742.000000,25533.000000,42225.000000,46138.00000
75%,1400.000000,1402.000000,26377.000000,26630.000000,52388.000000,55080.00000
max,1460.000000,1490.000000,34290.000000,34300.000000,109031.000000,106663.00000


Saving prepared data to csv file

In [12]:
prepared_meetings_csv_path = Path("data") / "meetings.csv"
meetings.to_csv(prepared_meetings_csv_path, index=False)
print("saved", prepared_meetings_csv_path)
print("shape", meetings.shape)
print(meetings["outcome"].value_counts().sort_index())

saved data/meetings.csv
shape (4184, 173)
outcome
1     690
2     839
3    1296
4    1359
Name: count, dtype: int64
